# pipeline_users

Silver users — special case not covered by pipeline_silver.ipynb.

Strategy: FULL OUTER JOIN of bronze.users_mongo + bronze.users_mssql on CPF (normalized).
Full refresh each run (~700 records). See Agent 6 spec in .claude/04_build.delegation.md.

In [ ]:
dbutils.widgets.removeAll()

dbutils.widgets.text("bronze_users_mongo",  "ubereats_dev.bronze.users_mongo")
dbutils.widgets.text("bronze_users_mssql",  "ubereats_dev.bronze.users_mssql")
dbutils.widgets.text("silver_users_table",  "ubereats_dev.silver.users")
dbutils.widgets.text("quarantine_table",    "ubereats_dev.quarantine.users")

In [ ]:
bronze_users_mongo  = dbutils.widgets.get("bronze_users_mongo")
bronze_users_mssql  = dbutils.widgets.get("bronze_users_mssql")
silver_users_table  = dbutils.widgets.get("silver_users_table")
quarantine_table    = dbutils.widgets.get("quarantine_table")

print(f"[users] mongo={bronze_users_mongo}  mssql={bronze_users_mssql}  "
      f"target={silver_users_table}  quarantine={quarantine_table}")

In [ ]:
# silver.users has no dedicated contract — schema defined here.
# CPF is the join key (normalized to 11 digits), Liquid Clustering by cpf (ADR-04).
# user_id added to enable join with search_events/recommendations in gold_user_behavior.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_users_table} (
        cpf              STRING NOT NULL,
        user_id          INT,
        uuid_mongo       STRING,
        uuid_mssql       STRING,
        email            STRING,
        first_name       STRING,
        last_name        STRING,
        phone_number     STRING,
        city             STRING,
        country          STRING,
        delivery_address STRING,
        birthday         DATE,
        job              STRING,
        company_name     STRING,
        _ingested_at     TIMESTAMP NOT NULL,
        _merged_at       TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (cpf)
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'delta.autoOptimize.optimizeWrite' = 'true'
    )
""")

# quarantine.users mirrors silver.users (same convention as the contract-driven
# domains, where quarantine mirrors silver — CLAUDE.md), plus the reason/ts pair.
# cpf has no NOT NULL here — that's exactly the condition that routes a row here.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table} (
        cpf                 STRING,
        user_id             INT,
        uuid_mongo          STRING,
        uuid_mssql          STRING,
        email               STRING,
        first_name          STRING,
        last_name           STRING,
        phone_number        STRING,
        city                STRING,
        country             STRING,
        delivery_address    STRING,
        birthday            DATE,
        job                 STRING,
        company_name        STRING,
        _ingested_at        TIMESTAMP,
        _quarantine_reason  STRING NOT NULL,
        _quarantine_ts      TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (cpf)
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'delta.autoOptimize.optimizeWrite' = 'true'
    )
""")

print(f"[users] tables ready: {silver_users_table}, {quarantine_table}")

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, current_timestamp, lit, regexp_replace, row_number, desc


def normalize_cpf(c):
    """'123.456.789-01' → '12345678901' (11-digit string)."""
    return regexp_replace(c, r'[.\-]', '')


def dedup_by_cpf(df):
    """Keep the latest row per cpf_key (highest __source_ts_ms)."""
    w = Window.partitionBy("cpf_key").orderBy(desc("__source_ts_ms"))
    return (
        df.withColumn("_rn", row_number().over(w))
        .filter(col("_rn") == 1)
        .drop("_rn")
    )


def to_quarantine_shape(df, source):
    """Project a single bronze frame (missing cpf) into the quarantine.users shape."""
    string_null = lit(None).cast("string")
    fields = {
        "cpf": col("cpf_key"),
        "user_id": col("user_id"),
        "uuid_mongo": col("uuid") if source == "mongo" else string_null,
        "uuid_mssql": col("uuid") if source == "mssql" else string_null,
        "email": col("email") if source == "mongo" else string_null,
        "first_name": col("first_name") if source == "mssql" else string_null,
        "last_name": col("last_name") if source == "mssql" else string_null,
        "phone_number": col("phone_number"),
        "city": col("city") if source == "mongo" else string_null,
        "country": col("country"),
        "delivery_address": col("delivery_address") if source == "mongo" else string_null,
        "birthday": col("birthday") if source == "mssql" else lit(None).cast("date"),
        "job": col("job") if source == "mssql" else string_null,
        "company_name": col("company_name") if source == "mssql" else string_null,
        "_ingested_at": col("_ingested_at"),
        "_quarantine_reason": lit("missing_cpf"),
        "_quarantine_ts": current_timestamp(),
    }
    return df.select(*[expr.alias(name) for name, expr in fields.items()])


mongo_raw = (
    spark.table(bronze_users_mongo)
    .filter(col("__op") != "d")
    .withColumn("cpf_key", normalize_cpf(col("cpf")))
)
mssql_raw = (
    spark.table(bronze_users_mssql)
    .filter(col("__op") != "d")
    .withColumn("cpf_key", normalize_cpf(col("cpf")))
)

quarantine_df = to_quarantine_shape(mongo_raw.filter(col("cpf_key").isNull()), "mongo").unionByName(
    to_quarantine_shape(mssql_raw.filter(col("cpf_key").isNull()), "mssql")
)
quarantine_count = quarantine_df.count()
if quarantine_count > 0:
    (
        quarantine_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(quarantine_table)
    )
print(f"[users] quarantined (missing cpf): {quarantine_count}")

mongo_df = dedup_by_cpf(mongo_raw.filter(col("cpf_key").isNotNull()))
mssql_df = dedup_by_cpf(mssql_raw.filter(col("cpf_key").isNotNull()))

print(f"[users] mongo rows: {mongo_df.count()}  mssql rows: {mssql_df.count()}")

In [ ]:
from pyspark.sql.functions import coalesce, current_timestamp

joined_df = mongo_df.alias("m").join(
    mssql_df.alias("s"),
    col("m.cpf_key") == col("s.cpf_key"),
    how="full_outer",
)

users_df = joined_df.select(
    coalesce(col("m.cpf_key"), col("s.cpf_key")).alias("cpf"),
    coalesce(col("m.user_id"), col("s.user_id")).alias("user_id"),
    col("m.uuid").alias("uuid_mongo"),
    col("s.uuid").alias("uuid_mssql"),
    col("m.email"),
    col("s.first_name"),
    col("s.last_name"),
    coalesce(col("m.phone_number"), col("s.phone_number")).alias("phone_number"),
    col("m.city"),
    coalesce(col("m.country"), col("s.country")).alias("country"),
    col("m.delivery_address"),
    col("s.birthday"),
    col("s.job"),
    col("s.company_name"),
    coalesce(col("m._ingested_at"), col("s._ingested_at")).alias("_ingested_at"),
    current_timestamp().alias("_merged_at"),
)

print(f"[users] joined rows: {users_df.count()}")

# user_id is carried for gold_user_behavior's join with search_events/recommendations,
# but cpf — not user_id — is silver.users's real identity (MERGE/cluster key, ADR-04).
# A duplicate user_id would fan out that Gold join and trigger
# DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE — see
# DESIGN_GOLD_DIMENSION_JOIN_INTEGRITY.md. silver.users is a full-refresh table (no
# incremental MERGE), so the check only needs to look within this run's batch, not
# against a prior state.
dup_user_ids = (
    users_df
    .filter(col("user_id").isNotNull())
    .groupBy("user_id")
    .count()
    .filter("count > 1")
    .select("user_id")
)

user_id_quarantine_df = (
    users_df
    .join(dup_user_ids, "user_id", "left_semi")
    .drop("_merged_at")
    .withColumn("_quarantine_reason", lit("duplicate_user_id"))
    .withColumn("_quarantine_ts", current_timestamp())
)
user_id_quarantine_count = user_id_quarantine_df.count()
if user_id_quarantine_count > 0:
    (
        user_id_quarantine_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(quarantine_table)
    )
print(f"[users] quarantined (duplicate user_id): {user_id_quarantine_count}")

users_df = users_df.join(dup_user_ids, "user_id", "left_anti")

In [ ]:
# Full refresh — overwrite Silver users on each run (~700 records, safe to replace)
(
    users_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_users_table)
)
print(f"[users] silver.users refreshed: {users_df.count()} records")